# Causal Inference ~ Group 10

## Problem 6 ~ Card and Krueger (1994) Difference-in-Differences

We replicate the Card and Krueger (1994) difference-in-differences analysis of the New Jersey minimum wage increase using the dataset `DinD_ex.dta`. The variables are:

- `fte` (Y): full-time-equivalent number of employees at a restaurant.
- `nj` (G): indicator for being in New Jersey (1) vs. Pennsylvania (0).
- `after` (T): indicator for the post-minimum-wage-change period (1) vs. pre (0).
- `njafter` (D): interaction = NJ * after, i.e. 1 for NJ restaurants observed after the change.
- `sheet`: unique restaurant identifier.

# Part A ~ Replicating the DiD Regression

We estimate the canonical DiD specification with robust (HC1) standard errors:

$$\text{fte}_{i,t} = \beta_0 + \beta_1 \cdot \text{nj}_i + \beta_2 \cdot \text{after}_t + \beta_3 \cdot \text{njafter}_{i,t} + \varepsilon_{i,t}$$

The DiD estimate of the average treatment effect on the treated is the coefficient $\beta_3$ on `njafter`.

In [1]:
import pandas as pd
import statsmodels.api as sm

# Load data
df = pd.read_stata("causal_data/DinD_ex.dta")

# DiD regression with robust standard errors
Y = df["fte"]
X = df[["nj", "after", "njafter"]]
X = sm.add_constant(X)

did_model = sm.OLS(Y, X, missing='drop').fit(cov_type='HC1')
print(did_model.summary())

# Explicit values referenced in the interpretation below
print("\n===== Key values referenced in interpretation =====")
print(f"Coefficient on njafter:  {did_model.params['njafter']:.4f}")
print(f"Robust SE on njafter:    {did_model.bse['njafter']:.4f}")
print(f"z-statistic on njafter:  {did_model.tvalues['njafter']:.4f}")
print(f"p-value on njafter:      {did_model.pvalues['njafter']:.4f}")
print(f"Coefficient on nj:       {did_model.params['nj']:.4f}")
print(f"Coefficient on after:    {did_model.params['after']:.4f}")
print(f"Constant:                {did_model.params['const']:.4f}")

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.315
Date:                Thu, 30 Apr 2026   Prob (F-statistic):              0.268
Time:                        21:17:01   Log-Likelihood:                -2519.3
No. Observations:                 698   AIC:                             5047.
Df Residuals:                     694   BIC:                             5065.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         20.3000      1.502     13.519      0.0